In [ ]:

from IPython.display import clear_output

%pip install catboost -q # Cat Boost must installed before import

clear_output()

In [ ]:
import kagglehub
import pandas as pd
import numpy as np


import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from catboost import CatBoostClassifier


In [ ]:
import kagglehub
# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")
print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
data_path = path + '/Q3_data.csv'
df = pd.read_csv(data_path)

In [ ]:
# Task 2: Write your code here:
df.head(5)

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Handle missing values appropriately:
missing_values = df.isnull().sum()
missing_values

In [ ]:
# Replace numbers by Median ** I saw by "Describe" the data hasnt "object" type **
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
df[num_cols] = SimpleImputer(strategy='median').fit_transform(df[num_cols])


In [ ]:
# Task 2: Check and remove duplicates if any exist:
df = df.drop_duplicates()

In [ ]:
# Task 3: Encode categorical variables if needed:
# There Is NO categorical variables

In [ ]:
# Task 4: Apply feature scaling to numerical features (Use StandardScaler):
X = df.drop(columns=['Target'])
y = df['Target']

scaler = StandardScaler()
X = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

In [ ]:
# Task 5: Check for target imbalance and state if it is imbalanced or not:
y.value_counts(normalize=True) # imbalanced



In [ ]:
# Task 1: Split the dataset into features (X) and target (y):
# I did it before Scaling

In [ ]:
# Task 2,3,4,5: Write your code here:

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_scores = []

for train_idx, val_idx in skf.split(X, y):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = CatBoostClassifier(
        iterations=200,
        depth=6,
        learning_rate=0.1,
        loss_function='Logloss',
        verbose=0,
        random_state=42
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)

    f1_scores.append(f1_score(y_val, y_pred))

print(f"Average F1 Score: {np.mean(f1_scores):.2f}")



In [ ]:
# Task 1: Plot feature importance from your trained model:
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.get_feature_importance()
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10,6))
plt.barh(
    feature_importance['Feature'][:15][::-1],
    feature_importance['Importance'][:15][::-1]
)
plt.xlabel("Importance Score")
plt.title("Top 15 Feature Importances")
plt.show()


In [ ]:
# Task 2: Identify and print the name of the most important feature (the 'golden feature'):
golden_feature = feature_importance.iloc[0]['Feature']
print("Golden Feature:", golden_feature)

In [ ]:
# Task Bonus: Write your code here: